# Real-World Accuracy Enhancement Pipeline
This pipeline focuses on extracting real-world accuracy through Best Practices: Feature Engineering, Cross-Validation, and XGBoost Hyperparameter Tuning.

In [24]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pickle

import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), 'src')))
from data_processing import clean_data

### 1. Data Loading & Preprocessing

In [25]:
df = pd.read_csv('Student Mental health.csv')

# Impute age median to avoid NaN drops
df['Age'] = df['Age'].fillna(df['Age'].median())

df = clean_data(df)
df.head()

,Choose your gender,Age,What is your CGPA?,Marital status,Do you have Depression?,Do you have Anxiety?,Do you have Panic attack?,Did you seek any specialist for a treatment?
0,1,20,3.750,1,1,0,1,0
1,0,24,3.750,0,0,0,1,0
2,0,21,3.245,0,0,1,0,0
3,0,21,3.750,0,1,1,1,0
4,1,23,0.995,1,1,0,1,0


### 2. Feature Engineering
We introduce `Symptom_Severity` to help the model quantify combined psychological strain before predicting depression.

In [26]:
X = df.drop('Do you have Depression?', axis=1)
y = df['Do you have Depression?']

X['Symptom_Severity'] = X['Do you have Anxiety?'] + X['Do you have Panic attack?']
X.head()

,Choose your gender,Age,What is your CGPA?,Marital status,Do you have Anxiety?,Do you have Panic attack?,Did you seek any specialist for a treatment?,Symptom_Severity
0,1,20,3.750,1,0,1,0,1
1,0,24,3.750,0,0,1,0,1
2,0,21,3.245,0,1,0,0,1
3,0,21,3.750,0,1,1,0,2
4,1,23,0.995,1,0,1,0,1


### 3. Scaling & Train-Test Split

In [27]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

### 4. Advanced Modeling (XGBoost) with GridSearchCV
Instead of blind guessing, we use Grid Search to cross-validate multiple hyperparameter combinations.

In [28]:
xgb_model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0]
}

grid_search = GridSearchCV(estimator=xgb_model, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
print("Optimal Settings Found:", grid_search.best_params_)

Optimal Settings Found: {'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 200, 'subsample': 1.0}


c:\Users\HP\miniconda3\Lib\site-packages\xgboost\training.py:199: UserWarning: [16:37:41] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


### 5. Evaluation & Export

In [29]:
y_pred = best_model.predict(X_test)
print(f"Test Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Test Accuracy: 0.7500

Classification Report:
               precision    recall  f1-score   support

           0       0.77      0.90      0.83       134
           1       0.69      0.44      0.54        66

    accuracy                           0.75       200
   macro avg       0.73      0.67      0.68       200
weighted avg       0.74      0.75      0.73       200



In [30]:
pickle.dump(best_model, open("mental_health_model.pkl", "wb"))
pickle.dump(scaler, open("scaler.pkl", "wb"))